In [87]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import TargetEncoder
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression  # Binary classifier for churn prediction
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv('../data/raw/telco-churn.csv')

df.head()

In [89]:
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output
# ------------------------
# 1. Basic Cleaning
# ------------------------

# Drop customerID if present
if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)

# Fix TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Convert Churn to 0/1
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# ------------------------
# 2. Split X and y
# ------------------------

X = df.drop("Churn", axis=1)
y = df["Churn"]

# ------------------------
# 3. Identify column types
# ------------------------

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

# print("Categorical columns:", cat_cols)
# print("Numerical columns:", num_cols)

# ------------------------
# 4. Preprocessing Pipeline (Target Encoding)
# ------------------------

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", TargetEncoder(smoothing=10), cat_cols)
])

# ------------------------
# 5. Train-test split
# ------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------------
# 6. Logistic Regression Model
# ------------------------

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",   # helps with churn imbalance
        solver="liblinear"
    ))
])

# ------------------------
# 7. Train Model
# ------------------------

pipe.fit(X_train, y_train)
y_pred_proba = pipe.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")

ROC AUC Score: 0.8401
